# Building your own container as Algorithm / Model Package


---

This notebook's CI test result for us-west-2 is as follows. CI test results in other regions can be found at the end of the notebook. 

![This us-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-2/build_and_train_models|sm-marketplace_building_your_own_container_as_package|sm-marketplace_building_your_own_container_as_package.ipynb)

---


With Amazon SageMaker, you can package your own algorithms that can than be trained and deployed in the SageMaker environment. This notebook will guide you through an example that shows you how to build a Docker container for SageMaker and use it for training and inference.

This is an extension of the [scikit-bring-your-own notebook](https://github.com/awslabs/amazon-sagemaker-examples/blob/master/advanced_functionality/scikit_bring_your_own/scikit_bring_your_own.ipynb). We append specific steps that help you create a new Algorithm / Model Package SageMaker entities, which can be sold on AWS Marketplace

By packaging an algorithm in a container, you can bring almost any code to the Amazon SageMaker environment, regardless of programming language, environment, framework, or dependencies. 

1. [Building your own algorithm container](#Building-your-own-algorithm-container)
    1. [When should I build my own algorithm container?](#When-should-I-build-my-own-algorithm-container?)
    1. [Permissions](#Permissions)
    1. [The example](#The-example)
    1. [The presentation](#The-presentation)
1. [Part 1: Packaging and Uploading your Algorithm for use with Amazon SageMaker](#Part-1:-Packaging-and-Uploading-your-Algorithm-for-use-with-Amazon-SageMaker)
    1. [An overview of Docker](#An-overview-of-Docker)
    1. [How Amazon SageMaker runs your Docker container](#How-Amazon-SageMaker-runs-your-Docker-container)
      1. [Running your container during training](#Running-your-container-during-training)
        1. [The input](#The-input)
        1. [The output](#The-output)
      1. [Running your container during hosting](#Running-your-container-during-hosting)
    1. [The parts of the sample container](#The-parts-of-the-sample-container)
    1. [The Dockerfile](#The-Dockerfile)
    1. [Building and registering the container](#Building-and-registering-the-container)
    1. [Testing your algorithm on your local machine or on an Amazon SageMaker notebook instance](#Testing-your-algorithm-on-your-local-machine-or-on-an-Amazon-SageMaker-notebook-instance)
1. [Part 2: Training and Hosting your Algorithm in Amazon SageMaker](#Part-2:-Training-and-Hosting-your-Algorithm-in-Amazon-SageMaker)
    1. [Set up the environment](#Set-up-the-environment)
    1. [Create the session](#Create-the-session)
    1. [Upload the data for training](#Upload-the-data-for-training)
    1. [Create an estimator and fit the model](#Create-an-estimator-and-fit-the-model)
    1. [Run a Batch Transform Job](#Batch-Transform-Job)
    1. [Deploy the model](#Deploy-the-model)
    1. [Optional cleanup](#Cleanup-Endpoint)
1. [Part 3: Package your resources as an Amazon SageMaker Algorithm](#Part-3---Package-your-resources-as-an-Amazon-SageMaker-Algorithm)
    1. [Algorithm Definition](#Algorithm-Definition)
1. [Part 4: Package your resources as an Amazon SageMaker ModelPackage](#Part-4---Package-your-resources-as-an-Amazon-SageMaker-ModelPackage)
    1. [Model Package Definition](#Model-Package-Definition)
1. [Part 5: Package your resources as an Amazon SageMaker Algorithm for Fine-tuning Large Models](#part-5---package-your-resources-as-an-amazon-sagemaker-algorithm-for-fine-tuning-large-models)
1. [Debugging Creation Issues](#Debugging-Creation-Issues)
1. [List on AWS Marketplace](#List-on-AWS-Marketplace)

## When should I build my own algorithm container?

You may not need to create a container to bring your own code to Amazon SageMaker. When you are using a framework (such as Apache MXNet or TensorFlow) that has direct support in SageMaker, you can simply supply the Python code that implements your algorithm using the SDK entry points for that framework. This set of frameworks is continually expanding, so we recommend that you check the current list if your algorithm is written in a common machine learning environment.

Even if there is direct SDK support for your environment or framework, you may find it more effective to build your own container. If the code that implements your algorithm is quite complex on its own or you need special additions to the framework, building your own container may be the right choice.

If there isn't direct SDK support for your environment, don't worry. You'll see in this walk-through that building your own container is quite straightforward.

## Permissions

Running this notebook requires permissions in addition to the normal `SageMakerFullAccess` permissions. This is because we'll creating new repositories in Amazon ECR. The easiest way to add these permissions is simply to add the managed policy `AmazonEC2ContainerRegistryFullAccess` to the role that you used to start your notebook instance. There's no need to restart your notebook instance when you do this, the new permissions will be available immediately.

## The example

Here, we'll show how to package a simple Python example which showcases the [decision tree][] algorithm from the widely used [scikit-learn][] machine learning package. The example is purposefully fairly trivial since the point is to show the surrounding structure that you'll want to add to your own code so you can train and host it in Amazon SageMaker.

The ideas shown here will work in any language or environment. You'll need to choose the right tools for your environment to serve HTTP requests for inference, but good HTTP environments are available in every language these days.

In this example, we use a single image to support training and hosting. This is easy because it means that we only need to manage one image and we can set it up to do everything. Sometimes you'll want separate images for training and hosting because they have different requirements. Just separate the parts discussed below into separate Dockerfiles and build two images. Choosing whether to have a single image or two images is really a matter of which is more convenient for you to develop and manage.

If you're only using Amazon SageMaker for training or hosting, but not both, there is no need to build the unused functionality into your container.

[scikit-learn]: http://scikit-learn.org/stable/
[decision tree]: http://scikit-learn.org/stable/modules/tree.html

## The presentation

This presentation is divided into two parts: _building_ the container and _using_ the container.

## Part 1: Packaging and Uploading your Algorithm for use with Amazon SageMaker

### An overview of Docker

If you're familiar with Docker already, you can skip ahead to the next section.

For many data scientists, Docker containers are a new concept, but they are not difficult, as you'll see here. 

Docker provides a simple way to package arbitrary code into an _image_ that is totally self-contained. Once you have an image, you can use Docker to run a _container_ based on that image. Running a container is just like running a program on the machine except that the container creates a fully self-contained environment for the program to run. Containers are isolated from each other and from the host environment, so the way you set up your program is the way it runs, no matter where you run it.

Docker is more powerful than environment managers like conda or virtualenv because (a) it is completely language independent and (b) it comprises your whole operating environment, including startup commands, environment variable, etc.

In some ways, a Docker container is like a virtual machine, but it is much lighter weight. For example, a program running in a container can start in less than a second and many containers can run on the same physical machine or virtual machine instance.

Docker uses a simple file called a `Dockerfile` to specify how the image is assembled. We'll see an example of that below. You can build your Docker images based on Docker images built by yourself or others, which can simplify things quite a bit.

Docker has become very popular in the programming and devops communities for its flexibility and well-defined specification of the code to be run. It is the underpinning of many services built in the past few years, such as [Amazon ECS].

Amazon SageMaker uses Docker to allow users to train and deploy arbitrary algorithms.

In Amazon SageMaker, Docker containers are invoked in a certain way for training and a slightly different way for hosting. The following sections outline how to build containers for the SageMaker environment.

Some helpful links:

* [Docker home page](http://www.docker.com)
* [Getting started with Docker](https://docs.docker.com/get-started/)
* [Dockerfile reference](https://docs.docker.com/engine/reference/builder/)
* [`docker run` reference](https://docs.docker.com/engine/reference/run/)

[Amazon ECS]: https://aws.amazon.com/ecs/

### How Amazon SageMaker runs your Docker container

Because you can run the same image in training or hosting, Amazon SageMaker runs your container with the argument `train` or `serve`. How your container processes this argument depends on the container:

* In the example here, we don't define an `ENTRYPOINT` in the Dockerfile so Docker will run the command `train` at training time and `serve` at serving time. In this example, we define these as executable Python scripts, but they could be any program that we want to start in that environment.
* If you specify a program as an `ENTRYPOINT` in the Dockerfile, that program will be run at startup and its first argument will be `train` or `serve`. The program can then look at that argument and decide what to do.
* If you are building separate containers for training and hosting (or building only for one or the other), you can define a program as an `ENTRYPOINT` in the Dockerfile and ignore (or verify) the first argument passed in. 

#### Running your container during training

When Amazon SageMaker runs training, your `train` script is run just like a regular Python program. A number of files are laid out for your use, under the `/opt/ml` directory:

    /opt/ml
    |-- input
    |   |-- config
    |   |   |-- hyperparameters.json
    |   |   `-- resourceConfig.json
    |   `-- data
    |       `-- <channel_name>
    |           `-- <input data>
    |-- model
    |   `-- <model files>
    `-- output
        `-- failure

##### The input

* `/opt/ml/input/config` contains information to control how your program runs. `hyperparameters.json` is a JSON-formatted dictionary of hyperparameter names to values. These values will always be strings, so you may need to convert them. `resourceConfig.json` is a JSON-formatted file that describes the network layout used for distributed training. Since scikit-learn doesn't support distributed training, we'll ignore it here.
* `/opt/ml/input/data/<channel_name>/` (for File mode) contains the input data for that channel. The channels are created based on the call to CreateTrainingJob but it's generally important that channels match what the algorithm expects. The files for each channel will be copied from S3 to this directory, preserving the tree structure indicated by the S3 key structure. 
* `/opt/ml/input/data/<channel_name>_<epoch_number>` (for Pipe mode) is the pipe for a given epoch. Epochs start at zero and go up by one each time you read them. There is no limit to the number of epochs that you can run, but you must close each pipe before reading the next epoch.

##### The output

* `/opt/ml/model/` is the directory where you write the model that your algorithm generates. Your model can be in any format that you want. It can be a single file or a whole directory tree. SageMaker will package any files in this directory into a compressed tar archive file. This file will be available at the S3 location returned in the `DescribeTrainingJob` result.
* `/opt/ml/output` is a directory where the algorithm can write a file `failure` that describes why the job failed. The contents of this file will be returned in the `FailureReason` field of the `DescribeTrainingJob` result. For jobs that succeed, there is no reason to write this file as it will be ignored.

#### Running your container during hosting

Hosting has a very different model than training because hosting is reponding to inference requests that come in via HTTP. In this example, we use our recommended Python serving stack to provide robust and scalable serving of inference requests:

![Request serving stack](images/stack.png)

This stack is implemented in the sample code here and you can mostly just leave it alone. 

Amazon SageMaker uses two URLs in the container:

* `/ping` will receive `GET` requests from the infrastructure. Your program returns 200 if the container is up and accepting requests.
* `/invocations` is the endpoint that receives client inference `POST` requests. The format of the request and the response is up to the algorithm. If the client supplied `ContentType` and `Accept` headers, these will be passed in as well. 

The container will have the model files in the same place they were written during training:

    /opt/ml
    `-- model
        `-- <model files>



### The parts of the sample container

In the `container` directory are all the components you need to package the sample algorithm for Amazon SageMager:

    .
    |-- Dockerfile
    |-- build_and_push.sh
    `-- decision_trees
        |-- nginx.conf
        |-- predictor.py
        |-- serve
        |-- train
        `-- wsgi.py

Let's discuss each of these in turn:

* __`Dockerfile`__ describes how to build your Docker container image. More details below.
* __`build_and_push.sh`__ is a script that uses the Dockerfile to build your container images and then pushes it to ECR. We'll invoke the commands directly later in this notebook, but you can just copy and run the script for your own algorithms.
* __`decision_trees`__ is the directory which contains the files that will be installed in the container.
* __`local_test`__ is a directory that shows how to test your new container on any computer that can run Docker, including an Amazon SageMaker notebook instance. Using this method, you can quickly iterate using small datasets to eliminate any structural bugs before you use the container with Amazon SageMaker. We'll walk through local testing later in this notebook.

In this simple application, we only install five files in the container. You may only need that many or, if you have many supporting routines, you may wish to install more. These five show the standard structure of our Python containers, although you are free to choose a different toolset and therefore could have a different layout. If you're writing in a different programming language, you'll certainly have a different layout depending on the frameworks and tools you choose.

The files that we'll put in the container are:

* __`nginx.conf`__ is the configuration file for the nginx front-end. Generally, you should be able to take this file as-is.
* __`predictor.py`__ is the program that actually implements the Flask web server and the decision tree predictions for this app. You'll want to customize the actual prediction parts to your application. Since this algorithm is simple, we do all the processing here in this file, but you may choose to have separate files for implementing your custom logic.
* __`serve`__ is the program started when the container is started for hosting. It simply launches the gunicorn server which runs multiple instances of the Flask app defined in `predictor.py`. You should be able to take this file as-is.
* __`train`__ is the program that is invoked when the container is run for training. You will modify this program to implement your training algorithm.
* __`wsgi.py`__ is a small wrapper used to invoke the Flask app. You should be able to take this file as-is.

In summary, the two files you will probably want to change for your application are `train` and `predictor.py`.

### The Dockerfile

The Dockerfile describes the image that we want to build. You can think of it as describing the complete operating system installation of the system that you want to run. A Docker container running is quite a bit lighter than a full operating system, however, because it takes advantage of Linux on the host machine for the basic operations. 

For the Python science stack, we will start from a standard Ubuntu installation and run the normal tools to install the things needed by scikit-learn. Finally, we add the code that implements our specific algorithm to the container and set up the right environment to run under.

Along the way, we clean up extra space. This makes the container smaller and faster to start.

Let's look at the Dockerfile for the example:

In [1]:
!cat container/Dockerfile

# Build an image that can do training and inference in SageMaker.
# This uses a Python 3 image with the nginx, gunicorn, flask stack
# for serving inferences in a stable way.

FROM python:3.10-slim-bullseye

LABEL maintainer="Amazon AI <sage-learner@amazon.com>"

# nginx + ca-certificates are needed for the serving stack. Python/pip come with the base image.
RUN apt-get -y update && apt-get install -y --no-install-recommends \
         nginx \
         ca-certificates \
    && rm -rf /var/lib/apt/lists/*

# Install the Python packages needed by scikit-learn training and the flask serving stack.
# pip leaves the install caches populated which uses a significant amount of space, so we
# disable the cache to keep the image small and fast to start.
RUN pip install --no-cache-dir numpy scipy scikit-learn pandas flask gevent gunicorn

# Set some environment variables. PYTHONUNBUFFERED keeps Python from buffering our standard
# output stream, which means that logs can be delivered to the user

### Building and registering the container

The following shell code shows how to build the container image using `docker build` and push the container image to ECR using `docker push`. This code is also available as the shell script `container/build-and-push.sh`, which you can run as `build-and-push.sh decision_trees_sample` to build the image `decision_trees_sample`. 

This code looks for an ECR repository in the account you're using and the current default region (if you're using an Amazon SageMaker notebook instance, this will be the region where the notebook instance was created). If the repository doesn't exist, the script will create it.

In [2]:
# [execution-copy] The container image is already built and pushed to ECR out-of-band
# (the original %%sh cell uses docker, which is unavailable in this papermill environment).
# The modernized Py3 image lives at <account>.dkr.ecr.us-west-1.amazonaws.com/decision-trees-sample:latest
print('Skipping in-notebook docker build; image already present in ECR.')


Skipping in-notebook docker build; image already present in ECR.


### Testing your algorithm on your local machine or on an Amazon SageMaker notebook instance

While you're first packaging an algorithm use with Amazon SageMaker, you probably want to test it yourself to make sure it's working right. In the directory `container/local_test`, there is a framework for doing this. It includes three shell scripts for running and using the container and a directory structure that mimics the one outlined above.

The scripts are:

* `train_local.sh`: Run this with the name of the image and it will run training on the local tree. You'll want to modify the directory `test_dir/input/data/...` to be set up with the correct channels and data for your algorithm. Also, you'll want to modify the file `input/config/hyperparameters.json` to have the hyperparameter settings that you want to test (as strings).
* `serve_local.sh`: Run this with the name of the image once you've trained the model and it should serve the model. It will run and wait for requests. Simply use the keyboard interrupt to stop it.
* `predict.sh`: Run this with the name of a payload file and (optionally) the HTTP content type you want. The content type will default to `text/csv`. For example, you can run `$ ./predict.sh payload.csv text/csv`.

The directories as shipped are set up to test the decision trees sample algorithm presented here.

## Part 2: Training, Batch Inference and Hosting your Algorithm in Amazon SageMaker

Once you have your container packaged, you can use it to train and serve models. Let's do that with the algorithm we made above.

### Set up the environment

Here we specify a bucket to use and the role that will be used for working with Amazon SageMaker.

In [3]:
# S3 prefix
common_prefix = "DEMO-scikit-byo-iris"
training_input_prefix = common_prefix + "/training-input-data"
batch_inference_input_prefix = common_prefix + "/batch-inference-input-data"

import os

# V3: get_execution_role moved to sagemaker-core session helper.
from sagemaker.core.helper.session_helper import get_execution_role

# [execution-copy] explicit role (get_execution_role only works on a SageMaker host)
role = "arn:aws:iam::729646638167:role/SageMakerRole"


### Create the session

The session remembers our connection parameters to Amazon SageMaker. We'll use it to perform all of our SageMaker operations.

In [4]:
# V3: use the sagemaker-core Session helper instead of sagemaker.Session().
import boto3
from sagemaker.core.helper.session_helper import Session

# [execution-copy] pin region to us-west-1
sess = Session(boto_session=boto3.Session(region_name="us-west-1"))

default_bucket = sess.default_bucket()
default_bucket_prefix = sess.default_bucket_prefix
default_bucket_prefix_path = ""

# If a default bucket prefix is specified, append it to the s3 path
if default_bucket_prefix:
    default_bucket_prefix_path = f"/{default_bucket_prefix}"
    common_prefix = f"{default_bucket_prefix}/{common_prefix}"
    training_input_prefix = f"{default_bucket_prefix}/{training_input_prefix}"
    batch_inference_input_prefix = f"{default_bucket_prefix}/{batch_inference_input_prefix}"


[07/15/26 11:31:41] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=12279968;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=12279969;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /Users/lucasjia/Library/Application Support/sagemaker/config.yaml


### Upload the data for training

When training large models with huge amounts of data, you'll typically use big data tools, like Amazon Athena, AWS Glue, or Amazon EMR, to create your data in S3. For the purposes of this example, we're using some the classic [Iris dataset](https://en.wikipedia.org/wiki/Iris_flower_data_set), which we have included. 

We can use use the tools provided by the Amazon SageMaker Python SDK to upload the data to a default bucket. 

In [5]:
TRAINING_WORKDIR = "data/training"

training_input = sess.upload_data(TRAINING_WORKDIR, key_prefix=training_input_prefix)
print("Training Data Location " + training_input)

Training Data Location s3://sagemaker-us-west-1-729646638167/DEMO-scikit-byo-iris/training-input-data


### Create an estimator and fit the model

In order to use Amazon SageMaker to fit our algorithm, we'll create an `Estimator` that defines how to use the container to train. This includes the configuration we need to invoke SageMaker training:

* The __container name__. This is constructed as in the shell commands above.
* The __role__. As defined above.
* The __instance count__ which is the number of machines to use for training.
* The __instance type__ which is the type of machine to use for training.
* The __output path__ determines where the model artifact will be written.
* The __session__ is the SageMaker session object that we defined above.

Then we use fit() on the estimator to train against the data that we uploaded above.

In [6]:
account = sess.boto_session.client("sts").get_caller_identity()["Account"]
region = sess.boto_session.region_name
image = "{}.dkr.ecr.{}.amazonaws.com/decision-trees-sample:latest".format(account, region)

In [7]:
# V3: the V2 Estimator is replaced by ModelTrainer. For a bring-your-own custom
# container, pass the ECR image via `training_image`; compute/output are structured configs.
from sagemaker.train import ModelTrainer
from sagemaker.core.training.configs import Compute, OutputDataConfig, InputData

tree = ModelTrainer(
    training_image=image,
    role=role,
    sagemaker_session=sess,
    compute=Compute(instance_type="ml.c4.2xlarge", instance_count=1),
    output_data_config=OutputDataConfig(
        s3_output_path="s3://{}{}/output".format(default_bucket, default_bucket_prefix_path)
    ),
    base_job_name="decision-trees-sample",
)

# The custom container expects a single channel named "training".
tree.train(
    input_data_config=[
        InputData(channel_name="training", data_source=training_input)
    ]
)

# Grab the completed training job + its model artifact for downstream batch transform / hosting.
training_job = tree._latest_training_job
model_data = training_job.model_artifacts.s3_model_artifacts
print("Model artifact:", model_data)

[07/15/26 11:31:42] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=12279974;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=12279975;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

[07/15/26 11:31:43] INFO     Role 'arn:aws:iam::729646638167:role/SageMakerRole' validated ]8;id=12279982;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=12279983;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/helper/iam_role_resolver.py#598\598]8;;\
                             for training. Using it.                                                               

                    INFO     StoppingCondition not provided. Using default:                         ]8;id=12279990;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-train/src/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=12279991;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-train/src/sagemaker/train/defaults.py#167\167]8;;\
                             max_runtime_in_seconds=3600 max_wait_time_in_seconds=None                             
                             max_pending_time_in_seconds=None                                                      

                    INFO     OutputDataConfig compression type not provided. Using default:         ]8;id=12279997;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-train/src/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=12279998;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-train/src/sagemaker/train/defaults.py#204\204]8;;\
                             GZIP                                                                                  

                    INFO     Training image URI:                                               ]8;id=12280005;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-train/src/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=12280006;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-train/src/sagemaker/train/model_trainer.py#558\558]8;;\
                             729646638167.dkr.ecr.us-west-1.amazonaws.com/decision-trees-sampl                     
                             e:latest                                                                              

...

[07/15/26 11:32:44] INFO     decision-trees-sample-20260715113143/algo-1-1784140336:             ]8;id=12280055;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=12280056;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#31585\31585]8;;\
                             Starting the training.                                                                

                    INFO     decision-trees-sample-20260715113143/algo-1-1784140336:             ]8;id=12280061;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=12280062;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#31585\31585]8;;\
                             validation-accuracy: 0.9533333333333334                                               

                    INFO     decision-trees-sample-20260715113143/algo-1-1784140336:             ]8;id=12280067;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=12280068;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#31585\31585]8;;\
                             Training complete.                                                                    

[07/15/26 11:32:54] INFO     Final Resource Status: Completed                                    ]8;id=12280074;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=12280075;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#31591\31591]8;;\

Model artifact: s3://sagemaker-us-west-1-729646638167/output/decision-trees-sample-20260715113143/output/model.tar.gz


### Batch Transform Job

Now let's use the model built to run a batch inference job and verify it works.


#### Batch Transform Input Preparation

The snippet below is removing the "label" column (column indexed at 0) and retaining the rest to be batch transform's input. 

NOTE: This is the same training data, which is a no-no from a statistical/ML science perspective. But the aim of this notebook is to demonstrate how things work end-to-end.

In [8]:
import pandas as pd

## Remove first column that contains the label
shape = pd.read_csv(TRAINING_WORKDIR + "/iris.csv", header=None).drop([0], axis=1)

TRANSFORM_WORKDIR = "data/transform"
shape.to_csv(TRANSFORM_WORKDIR + "/batchtransform_test.csv", index=False, header=False)

transform_input = (
    sess.upload_data(TRANSFORM_WORKDIR, key_prefix=batch_inference_input_prefix)
    + "/batchtransform_test.csv"
)
print("Transform input uploaded to " + transform_input)

Transform input uploaded to s3://sagemaker-us-west-1-729646638167/DEMO-scikit-byo-iris/batch-inference-input-data/batchtransform_test.csv


#### Run Batch Transform

Now that our batch transform input is setup, we run the transformation job next

In [9]:
# V3: batch transform is done via sagemaker-core resources. First register the trained
# artifact as a Model, then run a TransformJob against it.
import time
from sagemaker.core.resources import Model, TransformJob
from sagemaker.core.shapes import (
    ContainerDefinition,
    TransformInput,
    TransformOutput,
    TransformResources,
    TransformDataSource,
    TransformS3DataSource,
)

suffix = time.strftime("%Y-%m-%d-%H-%M-%S", time.gmtime())
transform_model_name = "decision-trees-batch-" + suffix
transform_job_name = "decision-trees-transform-" + suffix
transform_output_path = "s3://{}{}/transform-output".format(
    default_bucket, default_bucket_prefix_path
)

# Same custom image is used for training and inference.
Model.create(
    model_name=transform_model_name,
    primary_container=ContainerDefinition(image=image, model_data_url=model_data),
    execution_role_arn=role,
)

transform_job = TransformJob.create(
    transform_job_name=transform_job_name,
    model_name=transform_model_name,
    transform_input=TransformInput(
        data_source=TransformDataSource(
            s3_data_source=TransformS3DataSource(
                s3_data_type="S3Prefix", s3_uri=transform_input
            )
        ),
        content_type="text/csv",
        split_type="Line",
    ),
    transform_output=TransformOutput(s3_output_path=transform_output_path, assemble_with="Line"),
    transform_resources=TransformResources(instance_type="ml.m4.xlarge", instance_count=1),
)
transform_job.wait()

print("Batch Transform output saved to " + transform_output_path)

                    INFO     Creating model resource.                                            ]8;id=12281751;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=12281752;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#20593\20593]8;;\

[07/15/26 11:32:56] INFO     Creating transform_job resource.                                    ]8;id=12281758;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=12281759;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#32261\32261]8;;\

Output()

[07/15/26 11:37:58] INFO     Final Resource Status: Completed                                    ]8;id=12281765;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=12281766;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#32483\32483]8;;\

Batch Transform output saved to s3://sagemaker-us-west-1-729646638167/transform-output


##### Inspect the Batch Transform Output in S3

In [10]:
from urllib.parse import urlparse

parsed_url = urlparse(transform_output_path)
bucket_name = parsed_url.netloc
file_key = "{}/{}.out".format(parsed_url.path[1:], "batchtransform_test.csv")

s3_client = sess.boto_session.client("s3")

response = s3_client.get_object(Bucket=bucket_name, Key=file_key)
response_bytes = response["Body"].read().decode("utf-8")
print(response_bytes)

setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
virginica
virginica
virginica
virginica
virginica
virginica
virginica
virginica
virginica
virginica


### Deploy the model

Deploying the model to Amazon SageMaker hosting just requires a `deploy` call on the fitted model. This call takes an instance count, instance type, and optionally serializer and deserializer functions. These are used when the resulting predictor is created on the endpoint.

In [11]:
# V3: hosting is done with sagemaker-core resources (Model + EndpointConfig + Endpoint)
# instead of estimator.deploy(). The same custom image serves inference.
import time
from sagemaker.core.resources import Model, EndpointConfig, Endpoint
from sagemaker.core.shapes import ContainerDefinition, ProductionVariant

deploy_suffix = time.strftime("%Y-%m-%d-%H-%M-%S", time.gmtime())
model_name = "decision-trees-model-" + deploy_suffix
endpoint_config_name = "decision-trees-epc-" + deploy_suffix
endpoint_name = "decision-trees-ep-" + deploy_suffix

Model.create(
    model_name=model_name,
    primary_container=ContainerDefinition(image=image, model_data_url=model_data),
    execution_role_arn=role,
)

EndpointConfig.create(
    endpoint_config_name=endpoint_config_name,
    production_variants=[
        ProductionVariant(
            variant_name="AllTraffic",
            model_name=model_name,
            initial_instance_count=1,
            instance_type="ml.m4.xlarge",
            initial_variant_weight=1.0,
        )
    ],
)

predictor = Endpoint.create(
    endpoint_name=endpoint_name,
    endpoint_config_name=endpoint_config_name,
)
predictor.wait_for_status("InService")

                    INFO     Creating model resource.                                            ]8;id=12288881;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=12288882;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#20593\20593]8;;\

[07/15/26 11:38:01] INFO     Creating endpoint_config resource.                                  ]8;id=12288888;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=12288889;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#11069\11069]8;;\

                    INFO     Creating endpoint resource.                                         ]8;id=12288895;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=12288896;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#10228\10228]8;;\

Output()

[07/15/26 11:39:58] INFO     Final Resource Status: InService                                    ]8;id=12288902;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=12288903;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#10484\10484]8;;\

#### Choose some data and use it for a prediction

In order to do some predictions, we'll extract some of the data we used for training and do predictions against it. This is, of course, bad statistical practice, but a good way to see how the mechanism works.

In [12]:
shape = pd.read_csv(TRAINING_WORKDIR + "/iris.csv", header=None)

import itertools

a = [50 * i for i in range(3)]
b = [40 + i for i in range(10)]
indices = [i + j for i, j in itertools.product(a, b)]

test_data = shape.iloc[indices[:-1]]
test_X = test_data.iloc[:, 1:]
test_y = test_data.iloc[:, 0]

Prediction is as easy as calling predict with the predictor we got back from deploy and the data we want to do predictions with. The serializers take care of doing the data conversions for us.

In [13]:
# V3: invoke the endpoint via the core Endpoint resource. Serialize the feature rows to CSV
# (the same content type the custom container expects) and read the response body.
import io


def predict_csv(endpoint, values):
    buf = io.StringIO()
    for row in values:
        buf.write(",".join(str(v) for v in row) + "\n")
    response = endpoint.invoke(
        body=buf.getvalue().encode("utf-8"),
        content_type="text/csv",
        accept="text/csv",
    )
    payload = response.body.read()
    if isinstance(payload, bytes):
        payload = payload.decode("utf-8")
    return payload


print(predict_csv(predictor, test_X.values))

setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
virginica
virginica
virginica
virginica
virginica
virginica
virginica
virginica
virginica



#### Cleanup Endpoint

When you're done with the endpoint, you'll want to clean it up.

In [14]:
# V3: clean up the hosted endpoint, its config, and the model.
predictor.delete()
EndpointConfig.get(endpoint_config_name).delete()
Model.get(model_name).delete()

                    INFO     Deleting Endpoint - decision-trees-ep-2026-07-15-18-38-00           ]8;id=12291663;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=12291664;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#10428\10428]8;;\

                    INFO     Deleting EndpointConfig - decision-trees-epc-2026-07-15-18-38-00    ]8;id=12291670;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=12291671;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#11220\11220]8;;\

[07/15/26 11:40:00] INFO     Deleting Model - decision-trees-model-2026-07-15-18-38-00           ]8;id=12291677;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=12291678;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#20740\20740]8;;\

## Part 3 - Package your resources as an Amazon SageMaker Algorithm
(If you looking to sell a pretrained model (ModelPackage), please skip to Part 4. If you are looking to sell a large model (Algorithm), please skip to part 5.)

Now that you have verified that the algorithm code works for training, live inference and batch inference in the above sections, you can start packaging it up as an Amazon SageMaker Algorithm.

In [15]:
import boto3

smmp = boto3.client("sagemaker")

                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=12291683;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=12291684;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

### Algorithm Definition

SageMaker Algorithm is comprised of 2 parts:

1. A training image
2. An inference image (optional)

The key requirement is that the training and inference images (if provided) remain compatible with each other. Specifically, the model artifacts generated by the code in training image should be readable and compatible with the code in inference image. 

You can reuse the same image to perform both training and inference or you can choose to separate them. 


This sample notebook has already created a single algorithm image that perform both training and inference. This image has also been pushed to your ECR registry at {{image}}. You need to provide the following details as part of this algorithm specification:

#### Training Specification

You specify details pertinent to your training algorithm in this section.

#### Supported Hyper-parameters

This section captures the hyper-parameters your algorithm supports, their names, types, if they are required, default values, valid ranges etc. This serves both as documentation for buyers and is used by Amazon SageMaker to perform validations of buyer requests in the synchronous request path.

Please Note: While this section is optional, we strongly recommend you provide comprehensive information here to leverage our validations and serve as documentation. Additionally, without this being specified, customers cannot leverage your algorithm for Hyper-parameter tuning.

*** NOTE: The code below has hyper-parameters hard-coded in the json present in src/training_specification.py. Until we have better functionality to customize it, please update the json in that file appropriately***



In [16]:
from src.training_specification import TrainingSpecification
from src.training_channels import TrainingChannels
from src.metric_definitions import MetricDefinitions
from src.tuning_objectives import TuningObjectives
import json

training_specification = TrainingSpecification().get_training_specification_dict(
    ecr_image=image,
    supports_gpu=True,
    supported_channels=[
        TrainingChannels(
            "training",
            description="Input channel that provides training data",
            supported_content_types=["text/csv"],
        )
    ],
    supported_metrics=[MetricDefinitions("validation:accuracy", "validation-accuracy: (\\S+)")],
    supported_tuning_job_objective_metrics=[TuningObjectives("Maximize", "validation:accuracy")],
)

print(json.dumps(training_specification, indent=2, sort_keys=True))

{
  "TrainingSpecification": {
    "MetricDefinitions": [
      {
        "Name": "validation:accuracy",
        "Regex": "validation-accuracy: (\\S+)"
      }
    ],
    "SupportedHyperParameters": [
      {
        "DefaultValue": "100",
        "Description": "Grow a tree with max_leaf_nodes in best-first fashion. Best nodes are defined as relative reduction in impurity. If None then unlimited number of leaf nodes",
        "IsRequired": false,
        "IsTunable": true,
        "Name": "max_leaf_nodes",
        "Range": {
          "IntegerParameterRangeSpecification": {
            "MaxValue": "100000",
            "MinValue": "1"
          }
        },
        "Type": "Integer"
      }
    ],
    "SupportedTrainingInstanceTypes": [
      "ml.m4.xlarge",
      "ml.m4.2xlarge",
      "ml.m4.4xlarge",
      "ml.m4.10xlarge",
      "ml.m4.16xlarge",
      "ml.m5.large",
      "ml.m5.xlarge",
      "ml.m5.2xlarge",
      "ml.m5.4xlarge",
      "ml.m5.12xlarge",
      "ml.m5.24xlarge",

#### Inference Specification

You specify details pertinent to your inference code in this section. 


In [17]:
from src.inference_specification import InferenceSpecification
import json

inference_specification = InferenceSpecification().get_inference_specification_dict(
    ecr_image=image,
    supports_gpu=True,
    supported_content_types=["text/csv"],
    supported_mime_types=["text/csv"],
)

print(json.dumps(inference_specification, indent=4, sort_keys=True))

{
    "InferenceSpecification": {
        "Containers": [
            {
                "Image": "729646638167.dkr.ecr.us-west-1.amazonaws.com/decision-trees-sample:latest"
            }
        ],
        "SupportedContentTypes": [
            "text/csv"
        ],
        "SupportedRealtimeInferenceInstanceTypes": [
            "ml.m4.xlarge",
            "ml.m4.2xlarge",
            "ml.m4.4xlarge",
            "ml.m4.10xlarge",
            "ml.m4.16xlarge",
            "ml.m5.large",
            "ml.m5.xlarge",
            "ml.m5.2xlarge",
            "ml.m5.4xlarge",
            "ml.m5.12xlarge",
            "ml.m5.24xlarge",
            "ml.c4.xlarge",
            "ml.c4.2xlarge",
            "ml.c4.4xlarge",
            "ml.c4.8xlarge",
            "ml.c5.xlarge",
            "ml.c5.2xlarge",
            "ml.c5.4xlarge",
            "ml.c5.9xlarge",
            "ml.c5.18xlarge",
            "ml.p2.xlarge",
            "ml.p2.8xlarge",
            "ml.p2.16xlarge",
            "m

#### Validation Specification

In order to provide confidence to the sellers (and buyers) that the products work in Amazon SageMaker before listing them on AWS Marketplace, SageMaker needs to perform basic validations. The product can be listed in AWS Marketplace only if this validation process succeeds. This validation process uses the validation profile and sample data provided by you to run the following validations:

1. Create a training job in your account to verify your training image works with SageMaker.
2. Once the training job completes successfully, create a Model in your account using the algorithm's inference image and the model artifacts produced as part of the training job we ran. 
3. Create a transform job in your account using the above Model to verify your inference image works with SageMaker

In [18]:
from src.algorithm_validation_specification import AlgorithmValidationSpecification
import json

validation_specification = (
    AlgorithmValidationSpecification().get_algo_validation_specification_dict(
        validation_role=role,
        training_channel_name="training",
        training_input=training_input,
        batch_transform_input=transform_input,
        content_type="text/csv",
        instance_type="ml.c4.xlarge",
        output_s3_location="s3://{}/{}".format(default_bucket, common_prefix),
    )
)

print(json.dumps(validation_specification, indent=4, sort_keys=True))

{
    "ValidationSpecification": {
        "ValidationProfiles": [
            {
                "ProfileName": "ValidationProfile1",
                "TrainingJobDefinition": {
                    "HyperParameters": {},
                    "InputDataConfig": [
                        {
                            "ChannelName": "training",
                            "CompressionType": "None",
                            "ContentType": "text/csv",
                            "DataSource": {
                                "S3DataSource": {
                                    "S3DataDistributionType": "FullyReplicated",
                                    "S3DataType": "S3Prefix",
                                    "S3Uri": "s3://sagemaker-us-west-1-729646638167/DEMO-scikit-byo-iris/training-input-data"
                                }
                            },
                            "RecordWrapperType": "None"
                        }
                    ],
               

### Putting it all together

Now we put all the pieces together in the next cell and create an Amazon SageMaker Algorithm

In [19]:
import json
import time

algorithm_name = "scikit-decision-trees-" + str(round(time.time()))

create_algorithm_input_dict = {
    "AlgorithmName": algorithm_name,
    "AlgorithmDescription": "Decision trees using Scikit",
    "CertifyForMarketplace": True,
}
create_algorithm_input_dict.update(training_specification)
create_algorithm_input_dict.update(inference_specification)
create_algorithm_input_dict.update(validation_specification)

print(json.dumps(create_algorithm_input_dict, indent=4, sort_keys=True))

print("Now creating an algorithm in SageMaker")

smmp.create_algorithm(**create_algorithm_input_dict)

{
    "AlgorithmDescription": "Decision trees using Scikit",
    "AlgorithmName": "scikit-decision-trees-1784140800",
    "CertifyForMarketplace": true,
    "InferenceSpecification": {
        "Containers": [
            {
                "Image": "729646638167.dkr.ecr.us-west-1.amazonaws.com/decision-trees-sample:latest"
            }
        ],
        "SupportedContentTypes": [
            "text/csv"
        ],
        "SupportedRealtimeInferenceInstanceTypes": [
            "ml.m4.xlarge",
            "ml.m4.2xlarge",
            "ml.m4.4xlarge",
            "ml.m4.10xlarge",
            "ml.m4.16xlarge",
            "ml.m5.large",
            "ml.m5.xlarge",
            "ml.m5.2xlarge",
            "ml.m5.4xlarge",
            "ml.m5.12xlarge",
            "ml.m5.24xlarge",
            "ml.c4.xlarge",
            "ml.c4.2xlarge",
            "ml.c4.4xlarge",
            "ml.c4.8xlarge",
            "ml.c5.xlarge",
            "ml.c5.2xlarge",
            "ml.c5.4xlarge",
         

{'AlgorithmArn': 'arn:aws:sagemaker:us-west-1:729646638167:algorithm/scikit-decision-trees-1784140800',
 'ResponseMetadata': {'RequestId': 'e78642f3-88d9-4920-b4bd-0a5de1dba1b5',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'e78642f3-88d9-4920-b4bd-0a5de1dba1b5',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '102',
   'date': 'Wed, 15 Jul 2026 18:40:01 GMT'},
  'RetryAttempts': 0}}

#### Describe the algorithm

The next cell describes the Algorithm and waits until it reaches a terminal state (Completed or Failed)

In [20]:
import time
import json

while True:
    response = smmp.describe_algorithm(AlgorithmName=algorithm_name)
    status = response["AlgorithmStatus"]
    print(status)
    if status == "Completed" or status == "Failed":
        print(response["AlgorithmStatusDetails"])
        break
    time.sleep(5)

Pending


InProgress


InProgress


InProgress


InProgress


...

InProgress


InProgress


InProgress


InProgress


Completed
{'ValidationStatuses': [{'Name': 'ValidationProfile1', 'Status': 'Completed'}], 'ImageScanStatuses': [{'Name': '729646638167.dkr.ecr.us-west-1.amazonaws.com/decision-trees-sample@sha256:8c7c619b13feb6a1e4ab314b2d69d03619d63f15c0458ecc10d3b0cf0580f378', 'Status': 'Completed'}]}


## Part 4 - Package your resources as an Amazon SageMaker ModelPackage

In this section, we will see how you can package your artifacts (ECR image and the trained artifact from your previous training job) into a ModelPackage. Once you complete this, you can list your product as a pretrained model in the AWS Marketplace.

### Model Package Definition
A Model Package is a reusable model artifacts abstraction that packages all ingredients necessary for inference. It consists of an inference specification that defines the inference image to use along with an optional model weights location.


In [21]:
smmp = boto3.client("sagemaker")

#### Inference Specification

You specify details pertinent to your inference code in this section.


In [22]:
from src.inference_specification import InferenceSpecification

import json

modelpackage_inference_specification = InferenceSpecification().get_inference_specification_dict(
    ecr_image=image,
    supports_gpu=True,
    supported_content_types=["text/csv"],
    supported_mime_types=["text/csv"],
)

# Specify the model data resulting from the previously completed training job
modelpackage_inference_specification["InferenceSpecification"]["Containers"][0][
    "ModelDataUrl"
] = model_data
print(json.dumps(modelpackage_inference_specification, indent=4, sort_keys=True))

{
    "InferenceSpecification": {
        "Containers": [
            {
                "Image": "729646638167.dkr.ecr.us-west-1.amazonaws.com/decision-trees-sample:latest",
                "ModelDataUrl": "s3://sagemaker-us-west-1-729646638167/output/decision-trees-sample-20260715113143/output/model.tar.gz"
            }
        ],
        "SupportedContentTypes": [
            "text/csv"
        ],
        "SupportedRealtimeInferenceInstanceTypes": [
            "ml.m4.xlarge",
            "ml.m4.2xlarge",
            "ml.m4.4xlarge",
            "ml.m4.10xlarge",
            "ml.m4.16xlarge",
            "ml.m5.large",
            "ml.m5.xlarge",
            "ml.m5.2xlarge",
            "ml.m5.4xlarge",
            "ml.m5.12xlarge",
            "ml.m5.24xlarge",
            "ml.c4.xlarge",
            "ml.c4.2xlarge",
            "ml.c4.4xlarge",
            "ml.c4.8xlarge",
            "ml.c5.xlarge",
            "ml.c5.2xlarge",
            "ml.c5.4xlarge",
            "ml.c5.9xla

#### Validation Specification

In order to provide confidence to the sellers (and buyers) that the products work in Amazon SageMaker before listing them on AWS Marketplace, SageMaker needs to perform basic validations. The product can be listed in the AWS Marketplace only if this validation process succeeds. This validation process uses the validation profile and sample data provided by you to run the following validations:

* Create a transform job in your account using the above Model to verify your inference image works with SageMaker.


In [23]:
from src.modelpackage_validation_specification import ModelPackageValidationSpecification
import json

modelpackage_validation_specification = (
    ModelPackageValidationSpecification().get_validation_specification_dict(
        validation_role=role,
        batch_transform_input=transform_input,
        content_type="text/csv",
        instance_type="ml.c4.xlarge",
        output_s3_location="s3://{}/{}".format(default_bucket, common_prefix),
    )
)

print(json.dumps(modelpackage_validation_specification, indent=4, sort_keys=True))

{
    "ValidationSpecification": {
        "ValidationProfiles": [
            {
                "ProfileName": "ValidationProfile1",
                "TransformJobDefinition": {
                    "MaxConcurrentTransforms": 1,
                    "MaxPayloadInMB": 6,
                    "TransformInput": {
                        "CompressionType": "None",
                        "ContentType": "text/csv",
                        "DataSource": {
                            "S3DataSource": {
                                "S3DataType": "S3Prefix",
                                "S3Uri": "s3://sagemaker-us-west-1-729646638167/DEMO-scikit-byo-iris/batch-inference-input-data/batchtransform_test.csv"
                            }
                        },
                        "SplitType": "Line"
                    },
                    "TransformOutput": {
                        "Accept": "text/csv",
                        "AssembleWith": "Line",
                        "KmsKeyId

### Putting it all together

Now we put all the pieces together in the next cell and create an Amazon SageMaker Model Package.

In [24]:
import json
import time

model_package_name = "scikit-iris-detector-" + str(round(time.time()))
create_model_package_input_dict = {
    "ModelPackageName": model_package_name,
    "ModelPackageDescription": "Model to detect 3 different types of irises (Setosa, Versicolour, and Virginica)",
    "CertifyForMarketplace": True,
}
create_model_package_input_dict.update(modelpackage_inference_specification)
create_model_package_input_dict.update(modelpackage_validation_specification)
print(json.dumps(create_model_package_input_dict, indent=4, sort_keys=True))

smmp.create_model_package(**create_model_package_input_dict)

{
    "CertifyForMarketplace": true,
    "InferenceSpecification": {
        "Containers": [
            {
                "Image": "729646638167.dkr.ecr.us-west-1.amazonaws.com/decision-trees-sample:latest",
                "ModelDataUrl": "s3://sagemaker-us-west-1-729646638167/output/decision-trees-sample-20260715113143/output/model.tar.gz"
            }
        ],
        "SupportedContentTypes": [
            "text/csv"
        ],
        "SupportedRealtimeInferenceInstanceTypes": [
            "ml.m4.xlarge",
            "ml.m4.2xlarge",
            "ml.m4.4xlarge",
            "ml.m4.10xlarge",
            "ml.m4.16xlarge",
            "ml.m5.large",
            "ml.m5.xlarge",
            "ml.m5.2xlarge",
            "ml.m5.4xlarge",
            "ml.m5.12xlarge",
            "ml.m5.24xlarge",
            "ml.c4.xlarge",
            "ml.c4.2xlarge",
            "ml.c4.4xlarge",
            "ml.c4.8xlarge",
            "ml.c5.xlarge",
            "ml.c5.2xlarge",
            "ml.c

{'ModelPackageArn': 'arn:aws:sagemaker:us-west-1:729646638167:model-package/scikit-iris-detector-1784141275',
 'ResponseMetadata': {'RequestId': '9edae6de-2ad1-4cef-852d-3f6f05132f03',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '9edae6de-2ad1-4cef-852d-3f6f05132f03',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '108',
   'date': 'Wed, 15 Jul 2026 18:47:55 GMT'},
  'RetryAttempts': 0}}

#### Describe the ModelPackage 

The next cell describes the ModelPackage and waits until it reaches a terminal state (Completed or Failed)

In [25]:
import time
import json

while True:
    response = smmp.describe_model_package(ModelPackageName=model_package_name)
    status = response["ModelPackageStatus"]
    print(status)
    if status == "Completed" or status == "Failed":
        print(response["ModelPackageStatusDetails"])
        break
    time.sleep(5)

Pending


InProgress


InProgress


InProgress


InProgress


...

InProgress


InProgress


InProgress


InProgress


Completed
{'ValidationStatuses': [{'Name': 'ValidationProfile1', 'Status': 'Completed'}], 'ImageScanStatuses': [{'Name': '729646638167.dkr.ecr.us-west-1.amazonaws.com/decision-trees-sample@sha256:8c7c619b13feb6a1e4ab314b2d69d03619d63f15c0458ecc10d3b0cf0580f378', 'Status': 'Completed'}]}


## Part 5 - Package your resources as an Amazon SageMaker Algorithm for Fine-tuning Large Models
(Use this notebook to see how to onboard machine learning models where the weights can be >= 100 GB. For example, if you are listing a large language model, follow these instructions.)

Now that you have verified that the algorithm code works for training, live inference and batch inference in the above sections, you can start packaging it up as an Amazon SageMaker Algorithm.

### Create an Algorithm for Fine-tuning

You can use the [create_algorithm API](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/sagemaker/client/create_algorithm.html) to create fine-tuning algorithms. 

> Note: the new API field `AdditionalS3DataSource` under `TrainingSpecification` and `InferenceSpecification` that accept the base model weights from your S3 location. This allows you to provide larger sized model weights. 

**The Training and Inference containers also need to read the from the enviroment variable `SAGEMAKER_ADDITIONAL_S3_DATA_PATH` to access the pre-trained base model weights. This enviroment variable is the local path to the extracted artifacts in your container. You do not have the ability to customize the local path.**

> As of now, Gzip is the only supported CompressionType. S3Object is the only supported S3DataType. S3Uri must end with `.tar.gz` suffix.
Under `ValidationSpecification` , `ValidationRole`,  `TrainingJobDefinition` are required. `TransfromJobDefinition` is optional.

During Algorithm creation, a Training Job with `TrainingJobDefinition` setup will be run with `ValidationRole` to do basic validation on the training job container. If `TrainingJobDefinition` is provided in `ValidationSpecification`. During Algorithm creation,  Transform Job with `TransformJobDefinition` setup will be run with `ValidationRole` to do basic validation on the inference container. However, the transform inference Job run in this step will not have base weights downloaded to the container

**Thus, only the inference container needs to exit successfully without base weights or real fine-tuning.**


In [26]:
import boto3

smmp = boto3.client("sagemaker")

#### Create the Algorithm using Boto3

For an exhaustive list of parameters see the [CreateAlgorithm API](https://docs.aws.amazon.com/sagemaker/latest/APIReference/API_CreateAlgorithm.html).

The `AdditionalS3DataSource` artifacts are sent to your container. Your container can access them by reading the `SAGEMAKER_ADDITIONAL_S3_DATA_PATH` which contains the local path to the extracted artifacts. 

In [27]:
# NOTE (reference-only, not executed): fine-tuning large-model Algorithm creation template.
# This cell is illustrative boilerplate with <placeholder> values and is NOT meant to run.
# In V3 the sagemaker-core Session does not expose create_algorithm; use a boto3 SageMaker
# client (sess.boto_session.client("sagemaker")) to call the CreateAlgorithm API directly.
#
# sagemaker_client = sess.boto_session.client("sagemaker")
# response = sagemaker_client.create_algorithm(
#     AlgorithmName="<AlgorithmName>",
#     AlgorithmDescription="<AlgorithmName>",
#     TrainingSpecification={
#         "TrainingImage": "<ImageURI>",
#         # new field to pass inference base weights
#         "AdditionalS3DataSource": {
#             "CompressionType": "Gzip",
#             "S3DataType": "S3Object",
#             "S3Uri": "<S3LocationOfBaseWeights>",  # must end with .tar.gz suffix
#         },
#         "SupportedTrainingInstanceTypes": ["<specify compatible instance type>"],
#         "SupportedHyperParameters": [
#             {
#                 "Name": "<Name>",
#                 "Description": "<Description>",
#                 "Type": "String",  # Integer | Continuous | Categorical | FreeText
#                 "IsTunable": False,
#                 "IsRequired": False,
#             },
#         ],
#         "SupportsDistributedTraining": True,
#         "TrainingChannels": [
#             {
#                 "Name": "<Name>",
#                 "Description": "<Description>",
#                 "IsRequired": False,
#                 "SupportedContentTypes": ["application/json"],
#                 "SupportedCompressionTypes": ["None", "Gzip"],
#                 "SupportedInputModes": ["Pipe", "File"],
#             },
#         ],
#     },
#     InferenceSpecification={
#         "Containers": [  # Only Containers[0] can have AdditionalS3DataSource
#             {
#                 "Image": "<ImageURI>",
#                 "AdditionalS3DataSource": {
#                     "CompressionType": "Gzip",
#                     "S3DataType": "S3Object",
#                     "S3Uri": "<S3LocationOfBaseWeights>",  # must end with .tar.gz suffix
#                 },
#             }
#         ],
#         "SupportedTransformInstanceTypes": ["ml.c5.xlarge", "<specify compatible instance type>"],
#         "SupportedRealtimeInferenceInstanceTypes": ["<specify compatible instance type>"],
#         "SupportedContentTypes": [],
#         "SupportedResponseMIMETypes": ["application/json"],
#     },
#     ValidationSpecification={
#         "ValidationRole": "<RoleARN>",
#         "ValidationProfiles": [
#             {
#                 "ProfileName": "<ProfileName>",
#                 "TrainingJobDefinition": {
#                     "TrainingInputMode": "File",
#                     "HyperParameters": {"num_epochs": "1"},
#                     "InputDataConfig": [
#                         {
#                             "ChannelName": "<ChannelName>",
#                             "DataSource": {
#                                 "S3DataSource": {
#                                     "S3DataType": "S3Prefix",
#                                     "S3Uri": "<S3DatasetLocation>",
#                                     "S3DataDistributionType": "FullyReplicated",
#                                 }
#                             },
#                             "CompressionType": "None",
#                             "RecordWrapperType": "None",
#                         },
#                     ],
#                     "OutputDataConfig": {"KmsKeyId": "", "S3OutputPath": "<S3OutputPath>"},
#                     "ResourceConfig": {
#                         "InstanceType": "ml.m5.xlarge",
#                         "InstanceCount": 1,
#                         "VolumeSizeInGB": 10,
#                     },
#                     "StoppingCondition": {"MaxRuntimeInSeconds": 1800},
#                 },
#             }
#         ],
#     },
#     CertifyForMarketplace=True,
# )
print("Reference-only template cell; not executed.")

Reference-only template cell; not executed.


#### Testing the Fine-tuning Algorithm 

Once the Algorithm is created, you can follow the steps below to test the algorithm. 
> Note: that you can test it in the AWS account where the algorithm is created before listing it in the AWS Marketplace. 
Also, after you create a preview listing in the AWS Marketplace, you can verify it again from test accounts you specify in the listing, by subscribing to the listing and then follow the same steps.


#### Create Fine-Tuning training Job

Now you can use the Algorithm created for fine-tuning.
Specify the `AlgorithmName` under `AlgorithmSpecification` . Before training container starts, base weights file will be downloaded to training instances and extracted under the path indicated by environmental variable `SAGEMAKER_ADDITIONAL_S3_DATA_PATH`.

For example, base weights are specified at `s3://fine-tuning-models/model.tar.gz and model.tar.gz` which contains files `data1` and `data2` in the `model.tar.gz`. These files will be available under `$SAGEMAKER_ADDITIONAL_S3_DATA_PATH/data1` and `$SAGEMAKER_ADDITIONAL_S3_DATA_PATH/data2` in the container.

> Note: The `S3OutputPath` path will contain the fine-tuned weights in the end-user/customer's account. Make sure your Algorithm Training container does not output anything sensitive to both CloudWatch Logs (std_err/std_out in the container) or the fine-tuned model weights in S3 (artifacts saved in the container to `/opt/ml/model` during Training)

In [28]:
# NOTE (reference-only, not executed): fine-tuning training job template.
# In V3, either call the boto3 SageMaker CreateTrainingJob API through
# sess.boto_session.client("sagemaker").create_training_job(...), or use sagemaker-core
# TrainingJob.create(...) with an AlgorithmSpecification that references the algorithm name.
#
# sagemaker_client = sess.boto_session.client("sagemaker")
# response = sagemaker_client.create_training_job(
#     TrainingJobName="<TrainingJobName>",
#     RoleArn="<RoleArn>",
#     AlgorithmSpecification={
#         "AlgorithmName": "<AlgorithmName>",  # fine-tuning algorithm name
#         "TrainingInputMode": "File",
#     },
#     InputDataConfig=[
#         {
#             "ChannelName": "<ChannelName>",
#             "DataSource": {
#                 "S3DataSource": {
#                     "S3DataType": "S3Prefix",
#                     "S3Uri": "<S3DatasetLocation>",
#                     "S3DataDistributionType": "FullyReplicated",
#                 }
#             },
#             "CompressionType": "None",
#             "RecordWrapperType": "None",
#         },
#     ],
#     OutputDataConfig={
#         "S3OutputPath": "<S3OutputPath>",
#         # CompressionType Gzip so the output weights can be used for creating a model package
#         "CompressionType": "GZIP",
#     },
#     ResourceConfig={
#         "InstanceCount": 1,
#         "InstanceType": "ml.c4.xlarge",
#         "VolumeSizeInGB": 40,
#     },
#     StoppingCondition={"MaxRuntimeInSeconds": 3600},
# )
print("Reference-only template cell; not executed.")

Reference-only template cell; not executed.


Once training is completed, the fine-tuned weights will be uploaded to `OutputDataConfig:S3OutputPath`.


#### Create a Model Package

To create a model package resource that you can use to create deployable models in Amazon SageMaker.
You can do the following:


In [29]:
# NOTE (reference-only, not executed): create a ModelPackage from a fine-tuned algorithm.
# In V3, call the boto3 SageMaker CreateModelPackage API via
# sess.boto_session.client("sagemaker").create_model_package(...).
#
# sagemaker_client = sess.boto_session.client("sagemaker")
# model_package_name = "string"  # <= 63 characters
# create_model_pkg_rsp = sagemaker_client.create_model_package(
#     ModelPackageName=model_package_name,
#     SourceAlgorithmSpecification={
#         "SourceAlgorithms": [
#             {
#                 "AlgorithmName": "<AlgorithmName>",
#                 "ModelDataUrl": "<S3LocationOfFineTunedModelWeightsFromTrainingJob>",
#             }
#         ]
#     },
# )
# print(create_model_pkg_rsp)
# sagemaker_client.describe_model_package(ModelPackageName=model_package_name)
print("Reference-only template cell; not executed.")

Reference-only template cell; not executed.


#### Create SageMaker Model entity and Deploy to SageMaker Hosting

After the model package reaches a Completed status, you can use a model package to create a deployable model (SageMaker Model). You can then deploy this Model to am endpoint to get real-time inferences by creating a hosted endpoint (SageMaker Endpoint).

In [30]:
# NOTE (reference-only, not executed): create a SageMaker Model / EndpointConfig / Endpoint
# from the model package. In V3 use the boto3 SageMaker client, or the sagemaker-core
# Model / EndpointConfig / Endpoint resources shown in Part 2.
#
# sagemaker_client = sess.boto_session.client("sagemaker")
# create_model_rsp = sagemaker_client.create_model(
#     ModelName=model_package_name,
#     ExecutionRoleArn="<ExecutionRoleArn>",
#     PrimaryContainer={"ModelPackageName": model_package_name},
#     EnableNetworkIsolation=True,
# )
# create_endpoint_config_rsp = sagemaker_client.create_endpoint_config(
#     EndpointConfigName=model_package_name,
#     ProductionVariants=[
#         {
#             "VariantName": "<VariantName>",
#             "ModelName": model_package_name,
#             "InitialInstanceCount": 1,
#             "InstanceType": "ml.c5.xlarge",
#             "ContainerStartupHealthCheckTimeoutInSeconds": 3600,
#             "ModelDataDownloadTimeoutInSeconds": 3600,
#         }
#     ],
# )
# create_endpoint_rsp = sagemaker_client.create_endpoint(
#     EndpointName=model_package_name, EndpointConfigName=model_package_name
# )
# sagemaker_client.describe_endpoint(EndpointName=model_package_name)
print("Reference-only template cell; not executed.")

Reference-only template cell; not executed.


#### Invoke Endpoint for Inference

After the endpoint reaches InService status, you can invoke the endpoint for inference:

In [31]:
# NOTE (reference-only, not executed): invoke the fine-tuned endpoint for inference.
# In V3 use the boto3 sagemaker-runtime client, or the core Endpoint.invoke() shown in Part 2.
#
# sagemaker_runtime = sess.boto_session.client("sagemaker-runtime")
# rsp = sagemaker_runtime.invoke_endpoint(
#     EndpointName=model_package_name,
#     Body=b"<InferencePayload>",
#     ContentType="text/csv",  # any format your container supports
# )
# rsp["Body"].read()
print("Reference-only template cell; not executed.")

Reference-only template cell; not executed.


### Debugging Creation Issues

Entity creation typically never fails in the synchronous path. However, the validation process can fail for many reasons. If the above Algorithm creation fails, you can investigate the cause for the failure by looking at the "AlgorithmStatusDetails" field in the Algorithm object or "ModelPackageStatusDetails" field in the ModelPackage object. You can also look for the Training Jobs / Transform Jobs created in your account as part of our validation and inspect their logs for more hints on what went wrong. 

If all else fails, please contact AWS Customer Support for assistance!


### List on AWS Marketplace

Next, please go back to the Amazon SageMaker console, click on "Algorithms" (or "Model Packages") and you'll find the entity you created above. If it was successfully created and validated, you should be able to select the entity and "Publish new ML Marketplace listing" from SageMaker console.

<img src="images/publish-to-marketplace-action.png"/>


After creating and validating your fine-tuning Algorithms in Amazon SageMaker, you can list your products on AWS Marketplace. The listing process makes your products available for model consumers in the AWS Marketplace and the SageMaker console.

Detailed steps can be found here: https://docs.aws.amazon.com/marketplace/latest/userguide/ml-publishing-your-product-in-aws-marketplace.html


## Notebook CI Test Results

This notebook was tested in multiple regions. The test results are as follows, except for us-west-2 which is shown at the top of the notebook.

![This us-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-1/build_and_train_models|sm-marketplace_building_your_own_container_as_package|sm-marketplace_building_your_own_container_as_package.ipynb)

![This us-east-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-2/build_and_train_models|sm-marketplace_building_your_own_container_as_package|sm-marketplace_building_your_own_container_as_package.ipynb)

![This us-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-1/build_and_train_models|sm-marketplace_building_your_own_container_as_package|sm-marketplace_building_your_own_container_as_package.ipynb)

![This ca-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ca-central-1/build_and_train_models|sm-marketplace_building_your_own_container_as_package|sm-marketplace_building_your_own_container_as_package.ipynb)

![This sa-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/sa-east-1/build_and_train_models|sm-marketplace_building_your_own_container_as_package|sm-marketplace_building_your_own_container_as_package.ipynb)

![This eu-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-1/build_and_train_models|sm-marketplace_building_your_own_container_as_package|sm-marketplace_building_your_own_container_as_package.ipynb)

![This eu-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-2/build_and_train_models|sm-marketplace_building_your_own_container_as_package|sm-marketplace_building_your_own_container_as_package.ipynb)

![This eu-west-3 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-3/build_and_train_models|sm-marketplace_building_your_own_container_as_package|sm-marketplace_building_your_own_container_as_package.ipynb)

![This eu-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-central-1/build_and_train_models|sm-marketplace_building_your_own_container_as_package|sm-marketplace_building_your_own_container_as_package.ipynb)

![This eu-north-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-north-1/build_and_train_models|sm-marketplace_building_your_own_container_as_package|sm-marketplace_building_your_own_container_as_package.ipynb)

![This ap-southeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-1/build_and_train_models|sm-marketplace_building_your_own_container_as_package|sm-marketplace_building_your_own_container_as_package.ipynb)

![This ap-southeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-2/build_and_train_models|sm-marketplace_building_your_own_container_as_package|sm-marketplace_building_your_own_container_as_package.ipynb)

![This ap-northeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-1/build_and_train_models|sm-marketplace_building_your_own_container_as_package|sm-marketplace_building_your_own_container_as_package.ipynb)

![This ap-northeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-2/build_and_train_models|sm-marketplace_building_your_own_container_as_package|sm-marketplace_building_your_own_container_as_package.ipynb)

![This ap-south-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-south-1/build_and_train_models|sm-marketplace_building_your_own_container_as_package|sm-marketplace_building_your_own_container_as_package.ipynb)
